<table style="width:100%; border-bottom: 2px solid #ccc; margin-bottom: 20px;">
  <tr>
    <td style="vertical-align:middle;">
      <img src="../resources/ADI-Logo-RGB-FullColor.png" alt="Company Logo" height="30">
    </td>
    <td style="text-align:right; vertical-align:middle;">
      <p style="margin: 0;">Phased Array Systems</p>
      <p style="font-size: 14px; margin: 0;">Iain Derrington – ADEF Group, ADI</p>
      <p style="font-size: 12px; color: #555;">Field Applications & Platform Engineer</p>
    </td> 
  </tr>
</table>

In [ ]:
# Common Declarations and setup
import os
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from phaser_functions import *
from phaser_init import init_phaser_sdr

from adi import adf4159
from adi import ad9361
from adi import one_bit_adc_dac
from adi import ad9361
from adi.cn0566 import CN0566

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output 

# Get Script / Notebook root and full path to resources folder
phaser_root = get_phaser_root()
resource_path = phaser_root / "resources"

display(Markdown(f"Phaser root: **{phaser_root}**"))
display(Markdown(f"Resource path: **{resource_path}**\n"))

CN0566_IP_ADDRS   = "192.168.1.10"
PLUTO_IP_ADDRS    = "192.168.2.1"
FIELDFOX_IP_ADDRS = "192.168.1.30"

"""
This are default settings the following excercises.
"""
SIGNAL_FREQ = 100e3                      # 100 kHz Base Band Signal
TX_IF_FREQ  = int(2.1e9)                 # 2.1 GHz IF 
RX_IF_FREQ  = int(2.1e9)                 # 2.1 GHz IF 
RF_FREQ     = int(12.145e9)              # Target Tx Frequency = 12.145 GHz
SAMPLE_RATE = int(600e3)                 # SDR data rate = 600 ksps
SDR_BUF_SIZE= 1024 * 4
FFT_SIZE    = SDR_BUF_SIZE               # Buffer size (matches working script)
CHIRP_BW    = 500e6                      # Chirp BW 
CHIRP_T_US  = 500                        # Ramp Time (uS) - 0.5 ms
PFD_FREQ    = 100e6                      # ADF4159 internal PFD frequency




# Connect to each device on the phaser board.
# Note the adf4159 and one_bit_adc_dac are part of CN0566 object.
# We are instantiating directly for interest only
try:
    display(Markdown(f"Trying to connect to devices...\n"))
    
    pll = adf4159  (uri="ip:" + CN0566_IP_ADDRS)
    gpio = one_bit_adc_dac(uri="ip:" + CN0566_IP_ADDRS)
    sdr = ad9361   (uri="ip:" + PLUTO_IP_ADDRS)
    phaser = CN0566(uri="ip:"+ CN0566_IP_ADDRS)

    display(Markdown(f"✅Found ADF4159, GPIO pins, PlutoSDR and CN0566"))

except:
    display(Markdown(f"❌Unable to connect to CN0566 / ADF4159. Please check the IP addresses and connections."))
    print("")
    sys.exit(1)

# Set phaser.sdr to instance of PlutoSDR
phaser.sdr = sdr 

# Implementation of a Radar System

In the previous section, we introduced the fundamental operating principles of pulsed, CW, and FMCW radar systems.

In this section, we move from theory to practice by implementing these concepts using the CN0566 (Phaser) platform. The aim is to build intuition by working directly with real hardware and real signals.

We begin with CW radar, as it provides the simplest entry point and allows us to focus on hardware familiarisation and signal flow before introducing range processing.

---

## Part 1 — CW Radar (Doppler)

In this part, we will:

- Build a **CW X-band radar** capable of measuring Doppler frequency
- Become familiar with the **CN0566 (Phaser) hardware architecture**
- Learn how to configure and control Analog Devices RF components using **PyADI-IIO**
- Observe and interpret Doppler signatures from moving targets

---

## Part 2 — FMCW Radar (Range)

Building on the CW radar foundation, we will then extend the system to FMCW operation.

In this part, we will:

- Generate a **linear FMCW chirp** at the transmitter
- Receive and process the reflected signal
- Measure the **beat frequency** at the receiver
- Use the beat frequency to estimate the **range of a target**

---

## Learning Objectives

By the end of this section, you should understand:

- How radar theory maps onto real hardware constraints
- How timing, sampling rate, and RF configuration affect radar performance
- Why CW, FMCW, and TDD choices matter in practical radar systems

# Phaser Hardware Overview

As a reminder, the diagram below shows the high-level architecture of the **CN0566 (Phaser) board**:

<div style="text-align:center;">
  <img src="resources/blockdiagram.svg" alt="Phaser Block Diagram" height="300">
</div>

In addition to the Phaser board, the system uses a **PlutoSDR** as the RF transceiver and data interface.

The PlutoSDR is capable of:
- Generating and receiving **complex (I/Q) signals**
- Operating up to approximately **2 GHz RF**
- Supporting an instantaneous bandwidth (iBW) of up to **20 MHz**

Together, the PlutoSDR and Phaser form a flexible X-band radar platform where:
- The **PlutoSDR** handles baseband processing and RF up/downconversion
- The **Phaser** board provides beamforming, amplification, and antenna interfacing



# CW Radar
## CW Radar (Doppler)

The plan is to start at baseband and work our way through the system configuring for our use case. As with many of the IIO based demo's we need first configure the hardware for the particularly case. Then it'simply a matter of transmitting and receiving databuffers and performing all necessarry for the application at hand.

### Baseband + IF Signal Generation

Lets start with configuring the basedband and IF signal.
As per the diagram above, the PlutoSDR is the centre of this section.

We will generate **complex 100kHz sinusoidal signal** within Pyhon and use the Pluto SDR to shift this to an IF of **2.1GHz**.

Just as quick reminder of the architecture of the PlutoSDR:

<table style="width:100%; margin-bottom:20px; table-layout:auto;">
  <tr>
    <td style="vertical-align:middle; width:1%;">
      <img src="resources/pluto-block_diagram.png"
           alt="Pluto Block Diagram"
           style="max-width:300px; height:auto;">
    </td>
    <td style="vertical-align:middle;">
      <img src="resources/ad9361.svg"
           alt="AD9361 Block Diagram"
           style="max-width:800px; width:100%; height:auto;">
    </td>
  </tr>
</table>

As always, we're going to use Python and the pyadi-iio package to do the heavy lifting for us.
In the fisrt part of this notebook we have define some key parameters and instantiated the following objects:

```python
    pll = adf4159  (uri="ip:" + CN0566_IP_ADDRS)
    gpio = one_bit_adc_dac(uri="ip:" + CN0566_IP_ADDRS)
    sdr = ad9361   (uri="ip:" + PLUTO_IP_ADDRS)
    phaser = CN0566(uri="ip:"+ CN0566_IP_ADDRS)
```

We will now configure the Pluto (sdr) in this next section of code to generate the 2.1GHz IF signal. Whilst we are configuring the Tx side wil shall also configure the Rx side.

In [ ]:
# Generate a 10.5 GHz carrier wave on the output of the Phaser
display(Markdown("Running...\n"))

try: sdr.tx_destroy_buffer()
except: pass
try: sdr.rx_destroy_buffer()
except: pass


sdr.sample_rate           = SAMPLE_RATE   # 0.6 Msps
display(Markdown(f"Pluto Sample rate = {sdr.sample_rate /1e6} MHz"))

# -----------------------------
# Tx parameters
# -----------------------------
display(Markdown("**Tx Parameters**"))
sdr.tx_rf_bandwidth       = 4000000
display(Markdown(f"Pluto Tx Bandwidth = {sdr.tx_rf_bandwidth/1e6:.2f} MHz"))

sdr.tx_lo                 = TX_IF_FREQ    # 2.1 GHz Tx LO
display(Markdown(f"Tx Local Oscillsator = {sdr.tx_lo /1e6:.2f} MHz"))

sdr.tx_buffer_size = int(FFT_SIZE)
display(Markdown(f"Tx Local Oscillsator = {sdr.tx_buffer_size} bytes"))

sdr.tx_cyclic_buffer      = True          # Transmit continuously
display(Markdown(f"Tx Cylcing Buffer Enabled:  {sdr.tx_cyclic_buffer}"))

sdr.tx_hardwaregain_chan0 = -5            # Tx Gain
display(Markdown(f"Tx Chan0 gain = {sdr.tx_hardwaregain_chan0} dB"))

sdr.tx_hardwaregain_chan1 = -80            # Tx Gain
display(Markdown(f"Tx Chan1 gain = {sdr.tx_hardwaregain_chan1} dB"))

sdr.tx_enabled_channels   = [0]           # Ports are swapped. Channel routed to Tx2
display(Markdown(f"Tx Channel(s) enabled  = {sdr.tx_enabled_channels}\n"))


# -----------------------------
# Rx parameters
# -----------------------------
display(Markdown("**Rx Parameters**"))
sdr.rx_rf_bandwidth       = 4000000       # 4 MHz Bandwidth
display(Markdown(f"Pluto Rx Bandwidth = {sdr.rx_rf_bandwidth/1e6} MHz"))

sdr.rx_lo                 = RX_IF_FREQ    # 2.1 GHz Tx LO
display(Markdown(f"Tx Local Oscilator = {sdr.rx_lo /1e6:.2f} MHz"))

sdr.rx_buffer_size = int(FFT_SIZE)      # 1024 * 4
display(Markdown(f"Rx Buffer Size = {sdr.rx_buffer_size} bytes"))

sdr.rx_enabled_channels   = [0, 1] 
display(Markdown(f"Rx Channel(s) enabled  = {sdr.rx_enabled_channels}"))

sdr.gain_control_mode_chan0 = "manual"  # manual or slow_attack
display(Markdown(f"Gain Control Chan 0  = {sdr.gain_control_mode_chan0}"))

sdr.gain_control_mode_chan1 = "manual"  # manual or slow_attack
display(Markdown(f"Gain Control Chan 1  = {sdr.gain_control_mode_chan1}"))

sdr.rx_hardwaregain_chan0 = int(30)     # must be between -3 and 70
display(Markdown(f"Rx hardware gain chan 0  = {sdr.rx_hardwaregain_chan0}"))

sdr.rx_hardwaregain_chan1 = int(30)     # must be between -3 and 70
display(Markdown(f"Rx hardware gain chan 1  = {sdr.rx_hardwaregain_chan1}\n"))


# -----------------------------
# Baseband signal creation
# -----------------------------
fc = int(SIGNAL_FREQ / (SAMPLE_RATE / SDR_BUF_SIZE)) * (SAMPLE_RATE / SDR_BUF_SIZE)  # hit an integer bin to avoid leakage
ts = 1 / float(SAMPLE_RATE)

display(Markdown(f"**Constructing {fc/1000:.2f}kHz baseband signal**"))

N = FFT_SIZE
t = np.arange(0, N * ts, ts)
i = np.cos(2 * np.pi * t * fc) * 2 ** 14
q = np.sin(2 * np.pi * t * fc) * 2 ** 14
iq = i + 1j * q

sdr.tx(iq)

display(Markdown(f"SDR Transmitting @ {sdr.tx_lo/1e9:.2f} GHz"))
display(Markdown("✅Complete✅"))

### LO Generation and Frequency Planning

With the baseband and IF signal transmitting, we need to convert the signal to up X-Band. This is where the Rx Front end is designed to operate.
The phaser provides the necessary functionallity to provide both the Tx up conversion and Rx down conversion:

The block diagram below illustrates the LO generation and feedback path used within the Phaser board:

<div style="text-align:center;">
  <img src="resources/phaser-up-down-mixer-stage.svg" alt="Frequency Generation Block Diagram" width="00">
</div>


To generate a 10.5 GHz transmit signal, the on-board VCO must operate **2 GHz away** from the RF frequency. In this example, we choose:

$
\Large f_\text{VCO} = 10.5\,\text{GHz} + 2\,\text{GHz} = 12.5\,\text{GHz}
$

The VCO signal is fed back to the **[ADF4159](https://www.analog.com/en/products/adf4159.html)** PLL through a ÷4 divider, which must be accounted for when programming the PLL frequency.



The following sections detail how we use Python to setup LO and IF.

<details>
<summary><strong>LO Source and Routing Control (VCTRL)</strong></summary>
Two control lines, **VCTRL_1** and **VCTRL_2**, select the LO source and routing on the Phaser board.

Throughout this notebook, we use the **on-board LO**.

| VCTRL_1 | VCTRL_2 | LO Source   | LO Destination | Mixer (LTC5548) |
|--------:|--------:|------------|----------------|----------------|
| HIGH    | HIGH    | On-board LO | TX chain       | Disabled       |
| HIGH    | LOW     | On-board LO | LO_OUT SMA     | Enabled        |
| LOW     | HIGH    | External LO | TX chain       | Disabled       |
| LOW     | LOW     | External LO | LO_OUT SMA     | Enabled        |
</details>

#### IIO Control Objects Overview

If the code above executes successfully, we will have access to two Python control objects:

- **`myGPIO`**  
  Provides access to CN0566 board-level control pins.  
  This object allows us to **set and read GPIO control lines** such as VCTRL selection, TX/RX switching, divider enables, and board status signals.

- **`myADF4159`**  
  Represents the on-board **[ADF4159](https://www.analog.com/en/products/adf4159.html)** PLL and provides programmatic control of its configuration.  
  Through this object, we can configure both **continuous-wave (CW)** operation and **frequency-ramped (FMCW)** behaviour.

---

#### ADF4159 Exposed Properties

The table below lists the most relevant **[ADF4159](https://www.analog.com/en/products/adf4159.html)** properties exposed through pyadi-iio, along with a brief description of their function:


| Property | Description | Property | Description |
|---------|-------------|---------|-------------|
| `clk1_div_value` | Divider value for CLK1 output | `clk1_mode` | Operating mode for CLK1 |
| `clk2_div_value` | Divider value for CLK2 output | `delay_clk` | Clock source for ramp delay timing |
| `delay_start_en` | Enables delayed ramp start | `delay_word` | Delay value before ramp start |
| `enable` | Powers the PLL on or off | `freq_dev_range` | Total frequency deviation for a ramp |
| `freq_dev_step` | Frequency step size per ramp increment | `freq_dev_time` | Time per frequency step |
| `frequency` | PLL output frequency (CW mode) | `muxout_sel` | Signal routed to the MUXOUT pin |
| `phase_value` | Output phase offset | `ramp_delay_en` | Enables delay before ramp generation |
| `ramp_en` | Enables frequency ramp generation | `ramp_mode` | Ramp mode selection |
| `sing_ful_tri` | Single / full triangle ramp selection | `trig_delay_en` | Enables trigger delay |
| `tx_trig_en` | Enables external TX trigger |  |  |

A full list of supported devices and methods can be found in the pyadi-iio documentation:  
https://analogdevicesinc.github.io/pyadi-iio/index.html

### Setting Up the Transmitter (TX)

The **[ADF4159](https://www.analog.com/en/products/adf4159.html)** PLL, **[ADAR1000](https://www.analog.com/en/products/adar1000.html)** beamformer devices, along with the various GPIO control lines on the Phaser board, are managed by a Raspberry.

The Raspberry Pi exposes these devices over Ethernet using the IIO daemon (`iiod`), which runs as a background service and provides a network-accessible IIO context.

Using this interface, we can configure the hardware directly from Python.

Lets begin by configuring the **[ADF4159](https://www.analog.com/en/products/adf4159.html)** PLL to generate the LO

To do this, we first create instances of:

- An **ADF4159 control object**, and  
- a **GPIO control object** for the Phaser board.

#### LO Generation Context

The **[ADF4159](https://www.analog.com/en/products/adf4159.html)** is used in conjunction with:

- A 100 MHz reference clock, and  
- An external **[HMC735](https://www.analog.com/en/products/hmc735.html)** VCO operating in the 10.5–12.2 GHz range
to generate the local oscillator (LO) signal used by both the TX and RX chains.

The divide-by-4 output of the **[HMC735](https://www.analog.com/en/products/hmc735.html)** is routed back to the RF input of the **[ADF4159](https://www.analog.com/en/products/adf4159.html)** and forms the second input to the Phase Frequency Detector (PFD).


#### ADF4159 Initial Configuration

Much of the low-level **[ADF4159](https://www.analog.com/en/products/adf4159.html)** configuration is handled automatically through a combination of:

- default parameters defined in the **Linux device tree overlay**, and  
- logic implemented in the **Linux IIO kernel driver**.

As a result, the Python control code can remain relatively simple while still producing **correct and repeatable hardware behaviour**.

If you are interested in the initial configuration settings and how they are stored, expand the section below.

<details>
<summary><strong>ADF4159 Default Configuration (Device Tree & Driver)</strong></summary>

Many of the default **[ADF4159](https://www.analog.com/en/products/adf4159.html)** configuration parameters are defined in the Linux  
CN0566 device tree overlay:

https://github.com/thorenscientific/rpi_setup_stuff/blob/main/rpi-cn0566-overlay.dts

The following settings are extracted from the `adf4159@2` fragment.

---

**Identity / bus / clocks**
- `compatible = "adi,adf4159"`
- `reg = <2>` (SPI chip select index)
- `label = "pll0"`
- `spi-max-frequency = <3600000>` (3.6 MHz)
- `clocks = <&clkin>`
- `clock-names = "clkin"`
- `clock-output-names = "rf_out"`
- `#clock-cells = <0>`

---

**Power-up / PLL core**
- `adi,power-up-frequency-hz = 3000000000`
- `adi,charge-pump-current-microamp = 900`
- `adi,negative-bleed-current-microamp = 0`
- `adi,reference-div-factor = 1`
- `adi,phase = 0`
- `adi,muxout-select = 15`

---

**Clock / divider outputs and timing**
- `adi,clk1-div = 100`
- `adi,clk2-timer-div = 0`
- `adi,clk2-timer-div-2 = 0`
- `adi,clk-div-mode = 0`
- `adi,delay-start-word = 0`
- `adi,interrupt-mode-select = 0`

---

**Ramp / modulation related (configured but not enabled)**
- `adi,deviation = 1000`
- `adi,deviation-2 = 0`
- `adi,deviation-offset = 1`
- `adi,ramp-mode-select = 0`
- `adi,ramp-status-mode = 3`
- `adi,step-word = 0`
- `adi,step-word-2 = 0`

---

For full implementation details, the Linux driver source can be found here:  
https://github.com/analogdevicesinc/linux/blob/main/drivers/iio/frequency/adf4159.c

</details>


#### Configure the Local Oscillator (LO)

We now configure the local oscillator (LO) used to generate the **10.5 GHz X-band transmit signal**.

As discussed earlier, the LO is generated using the on-board **[ADF4159](https://www.analog.com/en/products/adf4159.html)** PLL in combination with the external VCO.  
The same LO is shared between the **TX and RX chains**, ensuring phase coherence — a key requirement for CW Doppler radar operation.

In this section, the ADF4159 is placed into a known **continuous-wave (CW)** state and programmed to generate the frequency required to produce a **10.5 GHz RF signal** after mixing with the PlutoSDR IF.

The code below:
- Forces the ADF4159 into a **non-ramped (CW) operating mode**
- Accounts for the **÷4 feedback divider** between the VCO and the ADF4159 RF input
- Computes and programs the required PLL frequency
- Provides clear visibility of the resulting frequency plan for verification

In [ ]:
# Example of ADF4149 Configuration

# Fixed hardware assumption (you already validated this)
PRESCALE = 4  # VCO -> ADF4159 RF input is /4


# Put PLL into known CW state
pll.ramp_mode      = "disabled"
pll.freq_dev_step  = 0
pll.freq_dev_range = 0
pll.freq_dev_time  = 0
pll.powerdown      = 0

# Optional Route LO to SMA
#gpio.gpio_vctrl_1 = 1
#gpio.gpio_vctrl_2 = 0

# Create widget objects
out = widgets.Output()
status = widgets.HTML()

def apply_settings(rf_ghz, if_ghz):
    global target_frequency, if_frequency

    target_frequency = rf_ghz * 1e9
    if_frequency     = if_ghz * 1e9

    pll_frequency = int((target_frequency + if_frequency) / PRESCALE)

    pll.frequency = pll_frequency

    status.value = (
        f"<b>LO Out</b> {target_frequency/1e9:.3f} GHz &nbsp;|&nbsp; "
        f"<b>IF In:</b> {if_frequency/1e9:.3f} GHz &nbsp;|&nbsp; "
        f"<b>ADF4159 RF_IN:</b> {pll_frequency/1e9:.3f} GHz &nbsp;|&nbsp; "
        f"<b>N:</b> {pll_frequency/100e6:.2f}"
    )

# Sliders
rf_slider = widgets.FloatSlider(
    value=10.5,
    min=8.0,
    max=13.0,
    step=0.5,
    description="RF (GHz)",
    continuous_update=False
)

if_slider = widgets.FloatSlider(
    value=2.1,
    min=0.0,
    max=2.2,
    step=0.1,
    description="IF (GHz)",
    continuous_update=False
)

def on_change(change):
    apply_settings(rf_slider.value, if_slider.value)

rf_slider.observe(on_change, names="value")
if_slider.observe(on_change, names="value")

display(rf_slider, if_slider, status)

# Apply once at startup
apply_settings(rf_slider.value, if_slider.value)


In [ ]:
# Route LO to Tx chain via mixer

gpio.gpio_vctrl_1 = 1        # Select local oscillator
gpio.gpio_vctrl_2 = 1        # Route to Tx chain
gpio.gpio_tx_sw = 0          # Select OUT2 SMA Connecotr

#### Generate the IF

The PlutoSDR is used to generate an intermediate frequency (IF) signal at **2.1 GHz**, which is then mixed on the Phaser board to produce the final **10.5 GHz X-band RF output**.

> ⚠️ **Note**  
> On the CN0566 platform, the default transmit signal from the PlutoSDR is routed to **TX2**, which corresponds to the **internal u.FL connector**.  
> This differs from a standard off-the-shelf PlutoSDR, where TX1 is typically used.

The code below connects to the PlutoSDR and performs the following steps:

- Sets the PlutoSDR TX LO to 2.1 GHz
- Sets the sample rate to 600 kSPS
  - This defines the data rate between the AD9361 and the FPGA
  - With a 2.1 GHz LO, the transmitted spectrum is limited to approximately 1999–2004 MHz
- Enables cyclic transmission
  - A DMA engine continuously feeds the TX DAC
  - The FPGA replays the buffer in a circular fashion
- Enables the transmit channel
  - The transmit buffer must be a 1-D complex array
  - The buffer length defines the waveform repetition period
- Generates a low-frequency complex baseband tone and transmits it

If the code executes correctly, a 10.5 GHz CW signal should be observable on one of the Phaser TX output ports, after mixing with the LO generated in the previous section.

  

### CW RADAR Doppler Demo

In this section, we configure the both the RADAR transmistter and receiver on the CN0566 (Phaser) platform.

In the sections steps, we were intentionally verbose and instantiated separate control objects for:
- the LO/PLL device (ADF41xx),
- the PlutoSDR, and
- the CN0566 board-level GPIO lines.

This method was shown to simply show how pyadi-iio objects are instantiaded and used.

For the remainder of this notebook, we will use the higher-level **`CN0566`** class provided by pyadi-iio, which combines these elements into a single control object. This makes the code cleaner, reduces boilerplate, and helps keep the focus on the radar signal chain rather than device bring-up.

The code below uses this `CN0566` object to configure the Phaser platform for both transmit and receive, operating at 10.5 GHz.

In [ ]:
# Configure Phaser for CW Radar

FFT_SIZE = 1024 * 16

print("Running...\n")

"""
 Initialise Phaser Board
"""
# By default device_mode is "rx"
phaser.configure(device_mode="rx")
phaser.load_gain_cal(filename= resource_path / "gain_cal_val.pkl")
phaser.load_phase_cal(filename= resource_path / "phase_cal_val.pkl")

for i in range(0, 8):
    phaser.set_chan_phase(i, 0)

gain_list = [8, 34, 84, 127, 127, 84, 34, 8]  # Blackman taper

for i in range(0, len(gain_list)):
    phaser.set_chan_gain(i, gain_list[i], apply_cal=True)

# print(f"Gain Cal = {phaser.gcal}")
# print(f"Phase Cal = {phaser.pcal}")

# Setup Raspberry Pi GPIO states
try:
    phaser._gpios.gpio_tx_sw = 0    # 0 = TX_OUT_2, 1 = TX_OUT_1
    phaser._gpios.gpio_vctrl_1 = 1  # 1=Use onboard PLL/LO source  (0=disable PLL and VCO, and set switch to use external LO input)
    phaser._gpios.gpio_vctrl_2 = 1  # 1=Send LO to transmit circuitry  (0=disable Tx path, and send LO to LO_OUT)
except:
    phaser.gpios.gpio_tx_sw = 0     # 0 = TX_OUT_2, 1 = TX_OUT_1
    phaser.gpios.gpio_vctrl_1 = 1   # 1=Use onboard PLL/LO source  (0=disable PLL and VCO, and set switch to use external LO input)
    phaser.gpios.gpio_vctrl_2 = 1   # 1=Send LO to transmit circuitry  (0=disable Tx path, and send LO to LO_OUT)

"""
 Initialise Pluto
"""
try: sdr.tx_destroy_buffer()
except: pass
try: sdr.rx_destroy_buffer()
except: pass


# Configure SDR Rx
sdr.sample_rate = SAMPLE_RATE           # 600 ksps
sdr.rx_lo = RX_IF_FREQ                  # 2.1 GHz
sdr.rx_enabled_channels = [0, 1]        # enable Rx1 (voltage0) and Rx2 (voltage1)
sdr.rx_buffer_size = int(FFT_SIZE)      # 1024 * 4
sdr.gain_control_mode_chan0 = "manual"  # manual or slow_attack
sdr.gain_control_mode_chan1 = "manual"  # manual or slow_attack
sdr.rx_hardwaregain_chan0 = int(30)     # must be between -3 and 70
sdr.rx_hardwaregain_chan1 = int(30)     # must be between -3 and 70

# Configure SDR Tx - Match FMCW configuration (uses channel 1)
sdr.tx_lo = int(RX_IF_FREQ)
sdr.tx_buffer_size = int(FFT_SIZE)
sdr.tx_enabled_channels = [0, 1]         # Enable both channels (like FMCW)
sdr.tx_cyclic_buffer = True              # must set cyclic buffer to true for the tdd burst mode.  Otherwise Tx will turn on and off randomly
sdr.tx_hardwaregain_chan0 = -88          # Channel 0 low power
sdr.tx_hardwaregain_chan1 = 0            # Channel 1 full power (THIS is the active channel)


"""
Initialise the Phaser LO
"""
phaser.frequency = (int(RX_IF_FREQ + RF_FREQ)) // 4  # PLL feedback via /4 VCO output
phaser.freq_dev_step = 5690
phaser.freq_dev_range = 0         # ramp BW
phaser.freq_dev_time = 0          # ramp time
phaser.powerdown = 0
phaser.ramp_mode = "disabled"              # Disable ramp for CW mode

print(f"VCO Output Frequeny = {phaser.frequency*4/1e9} GHz")
print(f"Converter Sample Rate = {sdr.sample_rate/1e6} Msps")
print(f"SDR Tx LO frequency = {sdr.tx_lo/1e9 :.4f} GHz")
print(f"SDR Rx LO frequency = {sdr.rx_lo/1e9 :.4f} GHz")
print(f"Tx Gain Chan0 = {sdr.tx_hardwaregain_chan0} dB (low)")
print(f"Tx Gain Chan1 = {sdr.tx_hardwaregain_chan1} dB (ACTIVE)")
print(f"Rx Gain = {sdr.rx_hardwaregain_chan0} dB")
print(f"Buffer size = {sdr.tx_buffer_size} bytes \n")

"""
Transmit a signal
"""

# Create a sinewave waveform
fc = int(SIGNAL_FREQ / (SAMPLE_RATE / N)) * (SAMPLE_RATE / N)   # Signal is in a FFT bin
print(f"Baseband Freq = {fc/1e3} kHz")

ts = 1 / float(SAMPLE_RATE)
t = np.arange(0, N * ts, ts)
i = np.cos(2 * np.pi * t * fc) * 2 ** 14
q = np.sin(2 * np.pi * t * fc) * 2 ** 14
iq = 1 * (i + 1j * q)

# Send data - transmit on both channels like FMCW (channel 1 is the active one)
sdr._ctx.set_timeout(0)
sdr.tx([iq * 0.5, iq])  # Channel 0 at half power, Channel 1 at full power

print(f"\n✓ Transmitting +{fc/1e3:.2f} kHz baseband tone")
print(f"✓ Complete")

The transmitter and receiver are now all setup. Just to recap:

* Baseband signal is a 100kHz tone generated by the PlutoSDR
* The ADF4159 + HMC735 are configured to generate a 10.5 GHz LO, which is mixed with the 2 GHz IF from the PlutoSDR to produce the final RF signal
* The receiver is configured to capture the reflected signal and measure the beat frequency, which contains information about the target's range (distance).

So lets run and capture the data and view. 


In [ ]:
# Capture and plot Rx data for CW Radar - ZOOMED 95-105 kHz
# --- settings ---
fs = float(sdr.sample_rate)
N  = int(sdr.rx_buffer_size)

# Tightly zoomed frequency range: 95 kHz to 105 kHz
freq_start = 95e3
freq_end = 105e3

N_UPDATES = 200
WF_ROWS   = 200

DB_FLOOR  = -90
DB_CEIL   = 0

"""
fftfreq: is used to compute the Discrete Fourier Transform (DFT) sample frequencies. 
This function returns an array of frequency bin centers in cycles per unit of the sample 
spacing, with zero at the start. The frequency unit depends on the sample spacing provided.
N = Number of samples
d = Sampling Rate

fftshift: shifts the zero-frequency component to the center of the spectrum
"""
freq = np.fft.fftshift(np.fft.fftfreq(N, d=1/fs))
w = np.blackman(N)
w_sum = np.sum(w)


"""
Create a boolean mask to select FFT frequency bins within the zoomed frequency range
"""
mz = (freq >= freq_start) & (freq <= freq_end)
freq_z = freq[mz]
Nz = freq_z.size

"""
Initialise the waterfall/image buffer with DB_FLOOR values:
WF_ROWS time rows by Nz frequency bins, all set to the noise floor.
"""
img = np.full((WF_ROWS, Nz), DB_FLOOR, dtype=float)

def measure_width(freq_full, mag_full, f0=100e3, span_hz=10e3):
    """
    Measure the peak frequency, 3dB bandwidth, and spectral width (sigma) 
    of a signal within a specified frequency range.
        freq_full: array of frequencies corresponding to the FFT bins
        mag_full: array of magnitudes corresponding to the FFT bins
        f0: center frequency to search around (default 100 kHz)
        span_hz: frequency range to search around f0 (default ±10 kHz)
    Returns:
        fpk: peak frequency (Hz)
        bw_3db: 3dB bandwidth (Hz)
        sigma: spectral width (Hz)
    """

    # Create a boolean mask to select FFT bins within the specified frequency range
    m = (freq_full > (f0 - span_hz)) & (freq_full < (f0 + span_hz))
    
    # Extract the frequencies and magnitudes for the selected bins
    f = freq_full[m]

    # Find the index of the peak magnitude and corresponding frequency
    a = mag_full[m]
    i0 = np.argmax(a)
    fpk = f[i0]

    # Calculate 3dB bandwidth
    p = a**2
    thr = 0.5 * p[i0]
    above = p >= thr
    if np.any(above):
        f_low  = f[np.argmax(above)]
        f_high = f[len(above) - 1 - np.argmax(above[::-1])]
        bw_3db = f_high - f_low
    else:
        bw_3db = np.nan

    # Calculate spectral width (sigma)
    P = p / np.sum(p)
    mu = np.sum(f * P)
    sigma = np.sqrt(np.sum(((f - mu) ** 2) * P))
    return fpk, bw_3db, sigma

# Make figure once (NO sharex) ---
fig, (ax_fft, ax_wf) = plt.subplots( 2, 1, figsize=(12, 7),     gridspec_kw={"height_ratios": [1, 1]})

# Add extra vertical gap so labels never collide
fig.subplots_adjust(hspace=0.35)

# FFT
(line,) = ax_fft.plot(freq_z/1e3, np.full_like(freq_z, DB_FLOOR), lw=1)
ax_fft.axvline(SIGNAL_FREQ/1e3, linestyle="--", alpha=0.6, label='100 kHz Expected')
ax_fft.set_ylabel("Magnitude (dB rel)")
ax_fft.set_xlabel("Frequency (kHz)")
ax_fft.set_ylim(DB_FLOOR, DB_CEIL)
ax_fft.set_xlim(freq_start/1e3, freq_end/1e3)
ax_fft.grid(True)
ax_fft.legend()

# Waterfall
im = ax_wf.imshow(
    img,
    aspect="auto",
    origin="upper",
    extent=[freq_start/1e3, freq_end/1e3, 0, WF_ROWS],
    vmin=DB_FLOOR,
    vmax=DB_CEIL,
)
ax_wf.set_title("Waterfall")
ax_wf.set_xlabel("Frequency (kHz)")
ax_wf.set_ylabel("Time (newest at top)")
ax_wf.axvline(SIGNAL_FREQ/1e3, linestyle="--", alpha=0.6, color='white')

# Add colorbar for the waterfall plot
cbar = fig.colorbar(im, ax=ax_wf, pad=0.01)
cbar.set_label("dB (rel)")

plt.show()

print(f"Frequency range: {freq_start/1e3:.0f} - {freq_end/1e3:.0f} kHz")
print(f"Capturing {N_UPDATES} frames...\n")

for k in range(N_UPDATES):
    # Read data from SDR
    d = sdr.rx()

    # Sum the two channels and take only the first N samples (in case buffer is larger than FFT size)
    x = (d[0] + d[1])[:N]

    # Subtract mean and apply window, then take FFT
    X = np.fft.fftshift(np.fft.fft((x - np.mean(x)) * w))
    
    # Normalize magnitude by window sum and prevent log of zero by flooring to a small value
    mag = np.abs(X) / w_sum
    mag = np.maximum(mag, 1e-20)

    # Convert magnitude to dB relative to the maximum value, and clip to floor and ceiling
    db = 20*np.log10(mag / np.max(mag))
    db = np.clip(db, DB_FLOOR, DB_CEIL)

    # Update the line plot with the new data for the selected frequency bins
    db_z = db[mz]
    line.set_ydata(db_z)

    # Update the waterfall image by rolling the existing data down and inserting the new row at the top
    img = np.roll(img, 1, axis=0)
    img[0, :] = db_z
    im.set_data(img)

    # Measure peak frequency, 3dB bandwidth, and spectral width, and update the title
    fpk, bw3, sig = measure_width(freq, mag, f0=SIGNAL_FREQ, span_hz=10e3)
    ax_fft.set_title(f"Peak {fpk:,.1f} Hz | BW3dB {bw3:,.1f} Hz | σ {sig:,.1f} Hz")

    clear_output(wait=True)
    display(fig)

plt.close(fig)


The above demonstrated the abillity to measure the frequency variation due to dopper, but as detailed in the previous notebook, without modulation we can't measure range. To measure range, we will frequency modulate the carrier with a linear chirp.

We could generate the linear chirp using the PlutoSDR, however, that will limit our BW to about 20MHz. As a reminder, range resolution is limited by BW:

$ \Large
\begin {aligned}
\Delta R &= \frac{c}{2B} \\
         &= \frac{3e^8}{2 \times 20e^6} \\
         &= 7.5m 
\end{aligned}
$ 

To get around the iBW limitiation of the PlutoSDR, will shall use the ADF4159 ramp function!!!

### FMCW RADAR Demo  
#### Configuring the LO Chirp

Let's target a **range resolution of approximately 30 cm**.  
For FMCW radar, range resolution is given by:

$
\Large
\begin{aligned}
B &= \frac{c}{2\Delta R} \\
  &= \frac{3\times10^8}{2 \times 0.3} \\  \\
  &= 500~\text{MHz}
\end{aligned}
$

Thus we need a system with a 500MHz iBW. Litterally an order of magnitude higher that what the PlutoSDR can achieve. 

The [ADF4159](https://www.analog.com/en/products/adf4159.html) has a LO ramp generation function designed to help with FMCW Radars.It can generates the LO chirp by sweeping from a start frequency $f_0$ to $(f_0 + B)$ over a ramp time $T$.

A faster ramp (smaller $T$) increases the beat frequency $f_b$, which can make
targets easier to separate in frequency. However, the ramp rate is constrained by
several practical limits:

- The beat frequency must remain within the receiver bandwidth and ADC sample rate
- The PLL must be able to track the ramp without excessive phase error
- Loop bandwidth must be sufficient to support the chirp slope

You've gone **too fast** when you observe:

- Beat tone broadening for a static target
- Range sidelobes that do not improve with windowing
- Measured chirp slope differing from the programmed slope
- Non-linear phase versus time in the de-chirped signal
- Target-dependent bias (near and far targets behave differently)

> **Note:** As a rule of thumb for FMCW designs,  
> **ramp update rate < PLL loop bandwidth**


#### Programming a 500 MHz, 0.5 ms Chirp on CN0566

Let's now configure the CN0566 to generate a 500 MHz FMCW chirp with a **0.5 ms ramp time**.

At a high level, the [ADF4159](https://www.analog.com/en/products/adf4159.html) generates a ramp by incrementing the PLL N-divider
at a fixed update rate. Ignoring the fractional-N details for now:

- Increasing the N-divider increases the VCO frequency
- The divider is incremented once per ramp update
- This continues until the programmed frequency deviation is reached


#### IIO Perspective: What Controls the Ramp?

From the IIO interface:

- **Frequency deviation** is set by  
  `.freq_dev_range`
- **Frequency increment per update** is set by  
  `.freq_dev_step`

What is not immediately obvious is how often the frequency update occurs.

#### Ramp Update Rate (ADF4159 Datasheet)

From the [ADF4159](https://www.analog.com/en/products/adf4159.html) datasheet, the time between frequency updates is:

$
\Large
T_{\text{update}} = \text{CLK}_1 \times \text{CLK}_2 \times \frac{1}{f_{\text{PFD}}}
$

For the CN0566:

- $f_{\text{PFD}} = 100~\text{MHz}$  
  (configured at boot via the device tree)
- Reading back the divider values:
  - $\text{CLK}_1 = 1$
  - $\text{CLK}_2 = 100$

Therefore:

$
\Large
T_{\text{update}} = 1 \times 100 \times \frac{1}{100\,\text{MHz}} = 1~\mu\text{s}
$

So the ramp is updated at 1 MHz (every 1 μs).

#### Calculating the Required Step Size

To achieve a 0.5 ms ramp:

$
\Large
N_{\text{steps}} = \frac{0.5~\text{ms}}{1~\mu\text{s}} = 500
$

The required frequency deviation in the ADF4159 **/4 programming domain** is:

$
\Large
\Delta f_{\text{/4}} = \frac{500~\text{MHz}}{4} = 125~\text{MHz}
$

Therefore, the required frequency step per update is:

$
\Large
\text{freq\_dev\_step} = \frac{125~\text{MHz}}{500} = 250{,}000~\text{Hz}
$



#### Final Programming Summary

To generate a **500 MHz FMCW chirp with a 0.5 ms ramp time** on CN0566:

- `freq_dev_range = 125e6`  (Hz, /4 domain)
- `freq_dev_step  = 250000`   (Hz, /4 domain)
- Ramp update rate = **1 MHz (1 μs per step)**

In [ ]:
# Setup FMCW Tx and Rx on the Phaser
print("Running...\n")

"""
 Initialise Pluto
"""
try: sdr.tx_destroy_buffer()
except: pass
try: sdr.rx_destroy_buffer()
except: pass


# Configure SDR Rx
sdr.sample_rate = int(SAMPLE_RATE)      # 600 ksps
sdr.rx_lo = int(RX_IF_FREQ)             # 
sdr.rx_enabled_channels = [0, 1]        # enable Rx1 (voltage0) and Rx2 (voltage1)
sdr.rx_buffer_size = int(FFT_SIZE)      # 4 x1024
sdr.gain_control_mode_chan0 = "manual"  # manual or slow_attack
sdr.gain_control_mode_chan1 = "manual"  # manual or slow_attack
sdr.rx_hardwaregain_chan0 = int(30)     # must be between -3 and 70
sdr.rx_hardwaregain_chan1 = int(30)     # must be between -3 and 70

# Configure SDR Tx
sdr.tx_lo = int(TX_IF_FREQ)
sdr.tx_buffer_size = int(FFT_SIZE)      # Set buffer size BEFORE enabling channels
sdr.tx_enabled_channels = [0, 1]        # Enable both TX channels
sdr.tx_cyclic_buffer = True             # must set cyclic buffer to true for the tdd burst mode.  Otherwise Tx will turn on and off randomly
sdr.tx_hardwaregain_chan0 = -88         # must be between 0 and -88
sdr.tx_hardwaregain_chan1 = 0           # must be between 0 and -88 - Use channel 1 at full power

""" 
1) Put PLL in a known state (disable ramp while programming)
"""
phaser.powerdown = 0                    # Make sure PLL isn't powered down
phaser.ramp_en = 0
phaser.ramp_mode = "disabled"

"""
2) Program frequency + ramp parameters (note the /4 domain)
"""
phaser.frequency      = int(RF_FREQ // 4)           # /4 programming domain  in Hz
phaser.freq_dev_range = int(CHIRP_BW // 4)          # total deviation in /4 domain in Hz

# calcalute the frequency deviation step size to gives is the correct ramp rate
t_update = phaser.clk1_div_value * phaser.clk2_div_value * 1/PFD_FREQ 
num_of_updates = (CHIRP_T_US / 1e6) / t_update
freq_dev_step_size = int(CHIRP_BW // 4) / num_of_updates

phaser.freq_dev_step  = int(freq_dev_step_size)    # The frequency step / 4 in Hz       
phaser.freq_dev_time = int(CHIRP_T_US)

"""
3) Optional: delay / trigger settings
"""
phaser.delay_word      = 4095           # Delay word for ramp timing
phaser.delay_clk       = "PFD"          # can be 'PFD' or 'PFD*CLK1'
phaser.delay_start_en  = 0              # delay start
phaser.ramp_delay_en   = 0              # delay between ramps
phaser.trig_delay_en   = 0              # triangle delay
phaser.sing_ful_tri    = 0              # full triangle enable/disable
phaser.tx_trig_en      = 0              # start a ramp with TXdata

"""
4) Enable triangular ramp
"""
phaser.ramp_mode = "continuous_triangular"
phaser.sing_ful_tri = 0

# Some CN0566 examples "latch" settings by writing enable last.
phaser.enable = 0

print(f"VCO Output Frequeny = {phaser.frequency*4/1e9} GHz")
print(f"VCO Ramp Time = {phaser.freq_dev_time}uS")
print(f"Phaser Ramp Mode = {phaser.ramp_mode}")
print(f"Converter Sample Rate = {sdr.sample_rate/1e6} Msps")
print(f"SDR Tx LO frequency = {sdr.tx_lo/1e9 :.4f} GHz")
print(f"SDR Rx LO frequency = {sdr.rx_lo/1e9 :.4f} GHz")
print(f"Tx Gain = 6")
print(f"Rx Gain =  6")
print(f"Buffer size = {sdr.tx_buffer_size} bytes \n")

print("Ramp Calcuations...")
print(f"Tupdate = clk1 [{ phaser.clk1_div_value}] x clk2 [{phaser.clk2_div_value}] x 1/pfd [{1/PFD_FREQ}] = { 1e6 * t_update}uS")
print(f"Number of updates = Ramp Time [{CHIRP_T_US}uS] / Tupdate [{ 1e6 * t_update}uS] = {int(num_of_updates)}")
print(f"Calculated frequency deviation step size = iBW [{CHIRP_BW/1e6} MHz / 4] / Number of updates [{int(num_of_updates)}] = {int(freq_dev_step_size)}\n")

print("Read back from Phaser (ADF4149)...")
print("ramp_mode:", phaser.ramp_mode)
print("freq_dev_range:", phaser.freq_dev_range/1e6, " MHz")
print("freq_dev_step:", phaser.freq_dev_step)
print("freq_dev_time:", phaser.freq_dev_time)
print("clk1_div_value:", phaser.clk1_div_value)
print("clk2_div_value:", phaser.clk2_div_value)


"""
Transmit a signal - FIXED to generate POSITIVE frequency tone at +100 kHz
"""
# Create a complex sinewave waveform at POSITIVE frequency
fc = int(SIGNAL_FREQ / (SAMPLE_RATE / 4096)) * (SAMPLE_RATE / 4096)   # Signal is in a FFT bin

ts = 1 / float(SAMPLE_RATE)
t = np.arange(0, FFT_SIZE * ts, ts)
A = 2 ** 14

# FIXED: Use POSITIVE sign to create +100 kHz tone (not -100 kHz)
i = np.cos(2 * np.pi * t * fc) * A
q = np.sin(2 * np.pi * t * fc) * A
iq = i + 1j * q  # This creates a POSITIVE frequency tone

# Transmit on both channels: low power on ch0, full power on ch1
sdr.tx([iq * 0.5, iq])

print(f"\n✓ Transmitting +{fc/1e3:.2f} kHz baseband tone (positive frequency)")
print(f"✓ Complete")

If you were to connect the transmit output ('OUT2') to the SA you should be able to observe the frequency content and the ramp time.

<details><summary><Strong>SA Settings</Strong></summary>
Determining ramp rate from a SA isn't initially obvious:

Use Zero Span 

Set center frequency ≈ mid-band
fc ≈ 10.75 GHz

Set Span = 0 Hz (Zero Span)
RBW: 10–100 kHz
Time span: a few ms

What you'll see:

Bursts of energy every time the chirp passes through fc

Time between bursts = half ramp period (for triangular)

From that:

𝑇=2×(burst spacing)
T=2×(burst spacing)
𝑘=𝐵𝑇
k=TB
</details>
<table style="width:100%; border-bottom: 2px solid #ccc; margin-bottom: 20px;">
  <tr>
    <td style="vertical-align:middle;"> <img src="resources/Chirp_Frequency.png" alt="SA Chirp Frequency" height="30"></td>
    <td style="vertical-align:middle;"> <img src="resources/Chirp_Time.png" alt="SA Chirp Time" height="30"></td> 
  </tr>
</table>


The plot on the left shows the spectral content and it clearly illustrates the 500MHz BW. The plot on right shows a zero-span measurement with spike spacing of **0.1 ms**, corresponding to a **1 ms triangular chirp period** (0.5 ms up + 0.5 ms down) and an effective ramp rate of:


$ \Large
\begin {aligned}
RampRate &= \frac{f_1 - f_0}{t} \\ \\
         &= \frac{(11\times 10^{9}) - (10.5\times 10^{9})}{0.5\times 10^{-3}} \\ \\
         &= 1 \times 10^{12} \text{ Hz/s}
\end{aligned}
$

Let's set the target at 1m and calculate the expected beat frequency:

$  \Large
\begin {aligned}
f_b &= \frac{R \cdot 2k}{c} \\
    &= \frac{1 \times 2 \cdot (1 \times 10^{12})} {3 \times 10^{8}} \\
    &= 6.67 \text{ kHz}
\end{aligned}
$

With a **0.5 ms ramp** (vs the previous 5 ms), we get **10x higher beat frequencies** for the same target range. This makes targets much easier to detect above the noise floor and separate from TX-RX coupling.

At this point the Transmitter is configured to generate a 500 MHz FMCW chirp with a **0.5 ms ramp time**. The receiver is configured to capture the reflected signal and measure the beat frequency, which contains information about the target's range (distance).

So as a very quick recap of where we are right now.

1) We have created a complex 100kHz sin wave
2) This signal is sent to the PlutoSDR which is configured to shift the signal up to and IF of 2.1GHz
3) The output of the PlutoSDR is mixed with a linear chirp generated by the PLL circuit on board the Phaser board to produce a 10GHz to 10.5GHz signal, that is transmitted from Tx2 of the phaser board.
4) The reflected signal, goes through the exact opposite process:
5) The RF ramp signal is mixed down to 2.1GHz and fed in the the two SDR inputs
6) The PlutoSDR mixes down to basedband.

<div style="text-align:center;">
  <img src="resources/fmcw-architecure.png" alt="FMCW architecure" width="800">
</div>

So lets run and capture the data and view. If we take the FFT (single sided). We should see a 100kHz signal and with a target at ~1m, we expect to see frequency variation of approximately **±6.67 kHz** around 100 kHz as we randomly sample up-chirp and down-chirp phases.

In [ ]:
# Simple capture and plot with spectrogram - ZOOMED to +100 kHz region
print("Capturing data...")

fs = int(sdr.sample_rate)
N = int(sdr.rx_buffer_size)

# ZOOMED: Focus on region around +100 kHz to see beat frequency shifts
freq_start = 50e3    # 50 kHz
freq_end = 150e3     # 150 kHz

# Capture settings
N_UPDATES = 200
num_slices = 100

# Frequency axis
freq = np.linspace(-fs / 2, fs / 2, int(N))

# Create mask for frequency region of interest
freq_mask = (freq >= freq_start) & (freq <= freq_end)
freq_display = freq[freq_mask]

# Spectrogram buffer
img_array = np.ones((num_slices, len(freq_display))) * (-100)

peak_freq_history = []

# Create figure once
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Top plot: FFT
line, = ax1.plot(freq_display/1e3, np.zeros_like(freq_display), 'y', linewidth=2, label='FFT')
vline_if = ax1.axvline(SIGNAL_FREQ/1e3, color='g', linestyle='--', linewidth=2, alpha=0.5, label='100 kHz IF (no target)')
vline_peak = ax1.axvline(SIGNAL_FREQ/1e3, color='m', linestyle='-', linewidth=2, alpha=0.8, label='Current Peak')
ax1.set_xlabel("Frequency (kHz)")
ax1.set_ylabel("Magnitude (dB)")
ax1.set_title("Received Signal - Beat Frequency Spectrum")
ax1.grid(True, alpha=0.3)
ax1.set_xlim(freq_start/1e3, freq_end/1e3)
ax1.set_ylim(-80, 0)
ax1.legend(loc='upper right')

# Bottom plot: Spectrogram (waterfall)
im = ax2.imshow(
    img_array,
    aspect='auto',
    origin='upper',
    extent=[freq_start/1e3, freq_end/1e3, 0, num_slices],
    vmin=-80,
    vmax=0,
    cmap='viridis',
    interpolation='nearest'
)
ax2.axvline(SIGNAL_FREQ/1e3, color='g', linestyle='--', linewidth=1, alpha=0.5)
ax2.set_xlabel("Frequency (kHz)")
ax2.set_ylabel("Time (newest at top)")
ax2.set_title("Waterfall - Watch the bright line(s) move as target moves")
ax2.set_xlim(freq_start/1e3, freq_end/1e3)

# Add colorbar
cbar = fig.colorbar(im, ax=ax2, pad=0.01)
cbar.set_label("dB")

plt.tight_layout()

print(f"Starting live capture ({N_UPDATES} frames)...")
print(f"Zoomed to {freq_start/1e3:.0f}-{freq_end/1e3:.0f} kHz")
print(f"Move a target to see beat frequency change!\n")

for k in range(N_UPDATES):
    # Capture data
    data = sdr.rx()
    data = data[0] + data[1]
    
    # Apply window and FFT
    win_funct = np.blackman(len(data))
    y = data * win_funct
    sp = np.absolute(np.fft.fft(y))
    sp = np.fft.fftshift(sp)
    
    # Convert to dB
    s_mag = np.abs(sp) / np.sum(win_funct)
    s_mag = np.maximum(s_mag, 10 ** (-15))
    s_dbfs = 20 * np.log10(s_mag / (2 ** 11))
    
    # Update FFT plot
    line.set_ydata(s_dbfs[freq_mask])
    
    # Update spectrogram
    img_array = np.roll(img_array, 1, axis=0)
    img_array[0] = s_dbfs[freq_mask]
    im.set_data(img_array)
    
    # Find peak in displayed region
    peak_idx = np.argmax(s_mag[freq_mask])
    peak_freq = freq_display[peak_idx]
    peak_freq_history.append(peak_freq/1e3)
    
    # Update peak line position
    vline_peak.set_xdata([peak_freq/1e3, peak_freq/1e3])
    
    # Calculate beat frequency (offset from 100 kHz IF)
    beat_freq = peak_freq - SIGNAL_FREQ
    
    # Update title with beat frequency info
    if len(peak_freq_history) > 10:
        recent = peak_freq_history[-50:]
        recent_beat = [(f - SIGNAL_FREQ/1e3) for f in recent]
        ax1.set_title(
            f"Peak: {peak_freq/1e3:.2f} kHz | Beat: {beat_freq/1e3:+.2f} kHz | "
            f"Range: {min(recent):.1f}-{max(recent):.1f} kHz | σ={np.std(recent):.2f} kHz"
        )
    
    # Redraw every 3 frames for smooth updates
    if k % 3 == 0:
        clear_output(wait=True)
        display(fig)

plt.close(fig)

# Summary
print(f"\n=== Results ===")
print(f"Peak frequency range: {min(peak_freq_history):.2f} to {max(peak_freq_history):.2f} kHz")
print(f"Beat frequency range: {(min(peak_freq_history) - SIGNAL_FREQ/1e3):.2f} to {(max(peak_freq_history) - SIGNAL_FREQ/1e3):.2f} kHz")
print(f"Std deviation: {np.std(peak_freq_history):.2f} kHz")
print(f"Mean peak: {np.mean(peak_freq_history):.2f} kHz")

beat_std = np.std([(f - SIGNAL_FREQ/1e3) for f in peak_freq_history])
print(f"\n✓ Signal detected at {np.mean(peak_freq_history):.2f} kHz")
if beat_std > 1:
    print(f"✓ Beat frequency variation: {beat_std:.2f} kHz - unsynchronized FMCW working!")
    print(f"  This demonstrates the timing synchronization problem!")
else:
    print("⚠️  No variation - target may be absent or coupling dominates")

### What Just Happened? Understanding What You See

Let's interpret the FFT plot and waterfall display to understand what's actually happening in this unsynchronized FMCW system.

---

#### What the Plots Show

**The FFT (top plot)** shows frequency content of the received signal:
- **Green dashed line at 100 kHz**: This is where we'd see a signal if there was **no target** (just TX-RX coupling with zero range)
- **The actual peak (yellow/magenta)**: The dominant signal we're actually receiving
- **Peak location wandering**: Notice the peak frequency moves around over time

**The Waterfall (bottom plot)** shows frequency content over time:
- **Bright vertical line(s)**: Strong signal(s) present in the received spectrum
- **Line position moving left/right**: Beat frequency is changing over time
- **Multiple bright lines**: You might see two peaks (from up-chirp and down-chirp)

---

#### Scenario 1: You See a Stable Peak Near 100 kHz

**What this means:**
- **TX-RX coupling (leakage) is dominating** the received signal
- The direct leakage path from TX to RX has essentially zero range, so the beat frequency $f_b \approx 0$
- Any target reflection is present, but it's **40-60 dB weaker** than the coupling signal
- The coupling signal "masks" the target — you only see the 100 kHz IF carrier

**Why this happens:**
In a continuously-transmitting FMCW radar with no TX/RX isolation, the transmitter is always on while we're trying to receive. It's like trying to listen to a quiet echo while someone is shouting next to you — the direct sound drowns out the reflection.

This is a **real limitation** in practical FMCW radar design: without proper TX/RX isolation (or time-division duplexing), strong nearby returns (coupling, clutter, ground reflections) mask distant targets.

---

#### Scenario 2: You See Two Peaks Wandering Around 100 kHz

**What this means:**
- You're seeing **both the up-chirp and down-chirp beat frequencies**
- The peaks appear at roughly **100 ± 6-7 kHz** for a target at ~1 m range
- The peak positions vary because we're randomly sampling different parts of the chirp cycle
- The waterfall shows the bright lines moving as the target moves

**Why you see two peaks:**

The ADF4159 is generating a **continuous triangular chirp** with no synchronization:
- **Up-ramp (0.5 ms)**: LO sweeps 10.5 → 11.0 GHz → beat frequency is 100 kHz **+ $f_b$**
- **Down-ramp (0.5 ms)**: LO sweeps 11.0 → 10.5 GHz → beat frequency is 100 kHz **− $f_b$**
- **Total chirp period: 1 ms**

Each time we call `sdr.rx()`, we capture a 4096-sample buffer (≈6.8 ms at 600 kSPS). This buffer contains **multiple chirp cycles** with random phase alignment:
- Sometimes we start capture during an up-chirp
- Sometimes during a down-chirp
- Sometimes across a transition

Because the capture timing is random relative to the chirp, the **relative power** in up-chirp vs down-chirp portions varies. Sometimes the FFT shows one peak more strongly, sometimes the other, sometimes both equally.

**For a 1 m target:**
$
\Large
f_b = \frac{2Rk}{c} = \frac{2 \times 1 \times 10^{12}}{3 \times 10^{8}} \approx 6.67\,\text{kHz}
$

So you'd expect to see peaks around:
- **Up-chirp**: 100 + 6.67 = **106.67 kHz**
- **Down-chirp**: 100 − 6.67 = **93.33 kHz**

Your plot shows peaks at roughly **88 kHz and 114 kHz**, giving a beat frequency spread of **±14 kHz** around 100 kHz.

---

#### The Core Problem: Unsynchronized Buffer Capture

**The fundamental issue** is that the chirp generation (ADF4159) and data capture (PlutoSDR) are **not synchronized**:

1. **The ADF4159 free-runs** — it continuously generates triangular chirps with no external trigger
2. **The PlutoSDR captures whenever we call `sdr.rx()`** — timing has variable latency and no relationship to the chirp phase
3. **Each capture samples a random portion of the chirp cycle** — sometimes up-ramp, sometimes down-ramp, sometimes both

Without knowing **when** the chirp started and **which direction** the LO was sweeping during our capture, we cannot:
- Reliably determine whether we're looking at an up-chirp or down-chirp return
- Convert beat frequency to range (we don't know if $f_{\text{beat}}$ = 100 + $f_b$ or 100 − $f_b$)
- Get consistent, repeatable measurements
- Separate multiple targets at different ranges

Even if we captured only one chirp direction, the **random phase alignment** means each measurement samples a different portion of the chirp, losing coherent integration benefits.

---

#### The Solution: Synchronized Buffer Triggering

To make FMCW radar work properly, we need **synchronized buffer triggering** to align the transmit chirp with receive capture:

1. **Synchronized start trigger** — chirp generation and RX capture buffer start together, so we **know** we're capturing from the beginning of a chirp
2. **Deterministic timing** — every capture starts at the same chirp phase
3. **Known chirp direction** — we control whether we capture up-ramp or down-ramp

Synchronized triggering provides:
- **Aligned buffers**: Transmit chirp and receive capture start simultaneously
- **Synchronized capture**: Every buffer starts at the same chirp phase
- **Known chirp direction**: We control whether we capture up-ramp or down-ramp
- **Repeatable measurements**: Beat frequency is stable and deterministic
- **Accurate range measurement**: We can reliably convert $f_{\text{beat}} \rightarrow$ range

Note: This is still a **continuous wave FMCW** system where transmit and receive happen simultaneously. The synchronization ensures we know *when* in the chirp cycle we're capturing, not that we're alternating between transmit and receive periods.

Without synchronized triggering, you're measuring echoes while not knowing when the chirp started — even if you can see the signal, you don't know how to interpret the timing.

---

**In the next notebook**, we'll implement proper buffer synchronization using PlutoSDR's `sync_start` mechanism and show how synchronized FMCW radar can accurately and repeatably measure range.